# Ajuste na Estratégia de Classificação

Durante o andamento dos experimentos, a gente percebeu que seria interessante testar uma nova abordagem para a variável alvo do projeto. Inicialmente, o modelo estava trabalhando com quatro classificações diferentes, mas agora a ideia é reorganizar os rótulos em apenas três categorias:

- `Adequada`
- `Atenção`
- `Crítica`

Essa mudança não foi feita de forma aleatória. Conforme os primeiros testes foram sendo analisados, ficou perceptível que algumas classes apresentavam comportamentos muito próximos entre si, o que acabava aumentando a dificuldade do modelo em identificar padrões realmente relevantes. Em problemas de classificação, quanto maior a quantidade de classes e mais parecidas elas forem, maior tende a ser a complexidade da separação entre elas.

Com isso, surgiu a necessidade de investigar se uma estrutura com menos categorias conseguiria melhorar a capacidade de generalização dos algoritmos, principalmente do Random Forest.

A principal hipótese aqui é que, reduzindo a quantidade de classes, o modelo consiga:

- identificar padrões de forma mais consistente
- reduzir ambiguidades entre classificações parecidas
- melhorar métricas de desempenho
- diminuir erros de confusão entre classes
- aumentar a estabilidade dos resultados durante os testes

Além disso, essa reorganização também permite analisar se o problema originalmente definido com quatro categorias estava introduzindo uma complexidade desnecessária para o cenário atual do projeto.

A partir dessa nova divisão, serão executados novos experimentos utilizando os mesmos algoritmos já trabalhados anteriormente, permitindo comparar diretamente os resultados obtidos com quatro classificações versus os resultados obtidos com apenas três.

Dessa forma, será possível avaliar de maneira mais concreta se a simplificação da variável alvo realmente contribui para melhorar a performance dos modelos e a qualidade geral das previsões.

# Preparação do Ambiente

In [ ]:

# IMPORTAÇÃO DE BIBLIOTECAS
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

SEED = 42


# DETECÇÃO DE AMBIENTE
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


# CARREGAMENTO DO DATASET
if IN_COLAB:
    print("Ambiente Google Colab detectado.")

    drive.mount('/content/drive')

    DATA_PATH = Path(
        "/content/drive/MyDrive/EDA_AquaSense/Dataset/raw/Combined Data/Combined_dataset.csv"
    )

else:
    print("Ambiente local/VS Code detectado.")

    DATA_PATH = Path("../../dataset/Combined_dataset.csv")


df = pd.read_csv(DATA_PATH)

print("Dataset carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")

df.head()

Ambiente Google Colab detectado.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset carregado com sucesso.
Shape do dataset: (2827977, 14)


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),Nitrogen (mg/l),Nitrate (mg/l),CCME_Values,CCME_WQI
0,Canada,SE649035-145565,River,12-01-1974,0.059248,1.30,8.1500,0.011917,8.07500,9.885,0.343917,11.73155,100.0,Excellent
1,Canada,SE649035-145565,River,12-01-1975,0.039821,1.38,7.8000,0.009417,7.73333,10.150,0.449083,11.82009,100.0,Excellent
2,Canada,SE649035-145565,River,12-01-1976,0.031341,2.23,7.8000,0.011000,7.46667,10.235,0.220750,14.87472,100.0,Excellent
3,Canada,SE649035-145565,River,12-01-1977,0.020501,1.61,8.1500,0.012333,7.78333,11.116,0.572250,15.89293,100.0,Excellent
4,Canada,SE649035-145565,River,12-01-1978,0.020023,1.64,4.3708,0.006182,7.10000,7.068,0.371091,15.22888,100.0,Excellent


# Conversão do Dataset para Formato Parquet

Após o carregamento do dataset original em formato CSV, será criada uma versão em formato Parquet para utilização nas próximas etapas do pipeline.

O formato Parquet é mais eficiente para armazenamento e leitura de grandes volumes de dados, além de preservar melhor os tipos das variáveis e reduzir o tamanho do arquivo. Sua utilização facilita operações futuras relacionadas à criação do rótulo, preparação dos dados, treinamento de modelos de Machine Learning e construção de dashboards.

In [ ]:
if IN_COLAB:
    parquet_path = Path(
        "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/water_quality.parquet"
    )
else:
    parquet_path = Path(
        "../../dataset/processed/water_quality.parquet"
    )

df.to_parquet(parquet_path, index=False)

print("Dataset convertido para Parquet com sucesso.")
print(f"Arquivo salvo em: {parquet_path}")

Dataset convertido para Parquet com sucesso.
Arquivo salvo em: /content/drive/MyDrive/EDA_AquaSense/Dataset/processed/water_quality.parquet


In [ ]:

df = pd.read_parquet(parquet_path)

print("Dataset Parquet carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")

df.head()

Dataset Parquet carregado com sucesso.
Shape do dataset: (2827977, 22)


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,CCME_Values,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status
0,Canada,SE649035-145565,River,12-01-1974,0.059248,1.30,8.1500,0.011917,8.07500,9.885,...,100.0,Excellent,1,1,1,0,1.0,1,4,Atenção
1,Canada,SE649035-145565,River,12-01-1975,0.039821,1.38,7.8000,0.009417,7.73333,10.150,...,100.0,Excellent,1,1,1,0,2.0,1,4,Atenção
2,Canada,SE649035-145565,River,12-01-1976,0.031341,2.23,7.8000,0.011000,7.46667,10.235,...,100.0,Excellent,1,1,1,0,3.7,1,4,Atenção
3,Canada,SE649035-145565,River,12-01-1977,0.020501,1.61,8.1500,0.012333,7.78333,11.116,...,100.0,Excellent,1,1,1,0,2.0,1,4,Atenção
4,Canada,SE649035-145565,River,12-01-1978,0.020023,1.64,4.3708,0.006182,7.10000,7.068,...,100.0,Excellent,1,0,1,0,3.7,1,3,Atenção


In [ ]:
# Listagem das colunas disponíveis
df.columns.tolist()

['Country',
 'Area',
 'Waterbody Type',
 'Date',
 'Ammonia (mg/l)',
 'Biochemical Oxygen Demand (mg/l)',
 'Dissolved Oxygen (mg/l)',
 'Orthophosphate (mg/l)',
 'pH (ph units)',
 'Temperature (cel)',
 'Nitrogen (mg/l)',
 'Nitrate (mg/l)',
 'CCME_Values',
 'CCME_WQI',
 'ph_ok',
 'od_ok',
 'dbo_ok',
 'nitrate_ok',
 'ammonia_limit',
 'ammonia_ok',
 'environmental_score',
 'conama_status']

In [ ]:
# Tipos de dados das colunas
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2827977 entries, 0 to 2827976
Data columns (total 14 columns):
 #   Column                            Dtype  
---  ------                            -----  
 0   Country                           object 
 1   Area                              object 
 2   Waterbody Type                    object 
 3   Date                              object 
 4   Ammonia (mg/l)                    float64
 5   Biochemical Oxygen Demand (mg/l)  float64
 6   Dissolved Oxygen (mg/l)           float64
 7   Orthophosphate (mg/l)             float64
 8   pH (ph units)                     float64
 9   Temperature (cel)                 float64
 10  Nitrogen (mg/l)                   float64
 11  Nitrate (mg/l)                    float64
 12  CCME_Values                       float64
 13  CCME_WQI                          object 
dtypes: float64(9), object(5)
memory usage: 302.1+ MB


# Validação dos Parâmetros Candidatos para o Rótulo CONAMA

Antes da construção da variável `conama_status`, foi realizada uma etapa de validação metodológica dos parâmetros presentes no dataset. O objetivo dessa análise foi verificar se as variáveis disponíveis possuem correspondência adequada com parâmetros ambientais utilizados na Resolução CONAMA nº 357/2005 e se apresentam documentação suficiente para utilização na construção do rótulo de qualidade da água.

Inicialmente, alguns parâmetros apresentavam ambiguidades relacionadas à forma química, unidade de medida e significado ambiental. Por esse motivo, foi realizada uma análise detalhada do artigo científico original associado ao dataset:

> *A Comprehensive Dataset of Surface Water Quality Spanning 1940–2023 for Empirical and ML Adopted Research*.

A análise do artigo permitiu identificar que os autores realizaram um processo de harmonização e padronização dos dados, incluindo:
- padronização de nomes de parâmetros;
- padronização de unidades de medida;
- conversão de formatos entre diferentes países;
- validação das variáveis utilizadas no dataset final.

Além disso, o artigo apresenta uma tabela oficial contendo a descrição semântica de cada variável e suas respectivas unidades de medida. Isso permitiu validar que os parâmetros utilizados possuem significado ambiental bem definido e podem ser comparados com indicadores clássicos de qualidade da água utilizados em normas ambientais e modelos de Water Quality Index (WQI).

A partir dessa validação documental, os parâmetros foram classificados em:
- parâmetros com correspondência direta com indicadores ambientais clássicos;
- parâmetros candidatos ao rótulo mediante análise normativa;
- parâmetros de uso mais contextual ou complementar.

Essa etapa é fundamental porque o rótulo `conama_status` será utilizado posteriormente como variável alvo dos modelos supervisionados de Machine Learning. Portanto, garantir coerência metodológica na escolha dos parâmetros aumenta a confiabilidade, interpretabilidade e justificativa científica do pipeline desenvolvido no projeto.

In [ ]:
parameter_validation = pd.DataFrame({

    "Coluna do Dataset": [
        "pH (ph units)",
        "Dissolved Oxygen (mg/l)",
        "Biochemical Oxygen Demand (mg/l)",
        "Nitrate (mg/l)",
        "Ammonia (mg/l)",
        "Nitrogen (mg/l)",
        "Orthophosphate (mg/l)",
        "Temperature (cel)"
    ],

    "Parâmetro Ambiental": [
        "pH",
        "Oxigênio Dissolvido",
        "Demanda Bioquímica de Oxigênio (DBO5)",
        "Nitrato (NO3⁻)",
        "Amônia (NH3)",
        "Nitrogênio Total",
        "Ortofosfato (PO4³⁻)",
        "Temperatura"
    ],

    "Descrição Validada no Artigo": [
        "Potencial hidrogeniônico da água",
        "Concentração de oxigênio dissolvido",
        "Quantidade de oxigênio necessária para decomposição biológica",
        "Concentração de nitrato",
        "Concentração de amônia",
        "Nitrogênio total presente",
        "Concentração de ortofosfato",
        "Temperatura da água"
    ],

    "Unidade": [
        "pH units",
        "mg/L",
        "mg/L",
        "mg/L",
        "mg/L",
        "mg/L",
        "mg/L",
        "°C"
    ],

    "Status de Validação": [
        "Correspondência direta",
        "Correspondência direta",
        "Correspondência direta",
        "Validado como candidato",
        "Validado como candidato",
        "Validado como candidato",
        "Validado com cautela",
        "Parâmetro contextual"
    ],

    "Decisão Inicial": [
        "Utilizar no rótulo",
        "Utilizar no rótulo",
        "Utilizar no rótulo",
        "Candidato ao rótulo",
        "Candidato ao rótulo",
        "Candidato ao rótulo",
        "Avaliar compatibilidade normativa",
        "Utilizar como variável contextual"
    ],

    "Justificativa": [
        "Parâmetro clássico de qualidade da água amplamente utilizado em normas ambientais.",
        "Indicador essencial da qualidade ecológica e da oxigenação da água.",
        "Indicador importante de poluição orgânica.",
        "Parâmetro relevante associado à presença de nutrientes e poluição.",
        "Parâmetro ambiental documentado e harmonizado no artigo.",
        "O artigo especifica que representa nitrogênio total presente na água.",
        "Apesar de validado no artigo, pode haver diferenças entre ortofosfato e fósforo total em normas ambientais.",
        "Influencia processos físico-químicos, mas não será utilizada diretamente na regra inicial do rótulo."
    ]

})

parameter_validation

,Coluna do Dataset,Parâmetro Ambiental,Descrição Validada no Artigo,Unidade,Status de Validação,Decisão Inicial,Justificativa
0,pH (ph units),pH,Potencial hidrogeniônico da água,pH units,Correspondência direta,Utilizar no rótulo,Parâmetro clássico de qualidade da água amplam...
1,Dissolved Oxygen (mg/l),Oxigênio Dissolvido,Concentração de oxigênio dissolvido,mg/L,Correspondência direta,Utilizar no rótulo,Indicador essencial da qualidade ecológica e d...
2,Biochemical Oxygen Demand (mg/l),Demanda Bioquímica de Oxigênio (DBO5),Quantidade de oxigênio necessária para decompo...,mg/L,Correspondência direta,Utilizar no rótulo,Indicador importante de poluição orgânica.
3,Nitrate (mg/l),Nitrato (NO3⁻),Concentração de nitrato,mg/L,Validado como candidato,Candidato ao rótulo,Parâmetro relevante associado à presença de nu...
4,Ammonia (mg/l),Amônia (NH3),Concentração de amônia,mg/L,Validado como candidato,Candidato ao rótulo,Parâmetro ambiental documentado e harmonizado ...
5,Nitrogen (mg/l),Nitrogênio Total,Nitrogênio total presente,mg/L,Validado como candidato,Candidato ao rótulo,O artigo especifica que representa nitrogênio ...
6,Orthophosphate (mg/l),Ortofosfato (PO4³⁻),Concentração de ortofosfato,mg/L,Validado com cautela,Avaliar compatibilidade normativa,"Apesar de validado no artigo, pode haver difer..."
7,Temperature (cel),Temperatura,Temperatura da água,°C,Parâmetro contextual,Utilizar como variável contextual,"Influencia processos físico-químicos, mas não ..."


# Definição dos Parâmetros Utilizados na Construção do Rótulo CONAMA

Após a etapa de validação metodológica dos parâmetros presentes no dataset, foi definida a seleção final das variáveis que serão utilizadas na construção do rótulo `conama_status`.

Essa definição foi realizada com base em:
- correspondência com parâmetros ambientais utilizados pela Resolução CONAMA nº 357/2005;
- documentação oficial presente no artigo científico do dataset;
- clareza semântica das variáveis;
- compatibilidade das unidades de medida;
- relevância ambiental dos parâmetros;
- coerência metodológica para construção do pipeline de Machine Learning.

Os parâmetros selecionados para a construção do rótulo foram:

- `pH (ph units)`
- `Dissolved Oxygen (mg/l)`
- `Biochemical Oxygen Demand (mg/l)`
- `Nitrate (mg/l)`
- `Ammonia (mg/l)`

Esses parâmetros foram escolhidos por apresentarem correspondência ambiental clara, forte relevância na avaliação da qualidade da água e compatibilidade suficiente com indicadores utilizados em normas ambientais e modelos de Water Quality Index (WQI).

Além disso, os parâmetros selecionados representam diferentes dimensões da qualidade da água:
- equilíbrio químico da água (`pH`);
- capacidade de oxigenação e suporte à vida aquática (`Dissolved Oxygen`);
- presença de poluição orgânica (`Biochemical Oxygen Demand`);
- concentração de nutrientes e contaminação nitrogenada (`Nitrate` e `Ammonia`).

Por outro lado, alguns parâmetros foram mantidos fora da construção inicial do rótulo:

- `Nitrogen (mg/l)`
- `Orthophosphate (mg/l)`
- `Temperature (cel)`

A variável `Nitrogen (mg/l)` não será utilizada porque o dataset já possui parâmetros nitrogenados mais específicos (`Nitrate` e `Ammonia`). Dessa forma, incluir nitrogênio total poderia gerar redundância entre indicadores e aumentar ambiguidades metodológicas durante a classificação ambiental.

A variável `Orthophosphate (mg/l)` também não será utilizada diretamente na regra do rótulo. Apesar de possuir relevância ambiental, o ortofosfato representa apenas uma fração do fósforo presente na água, enquanto normas ambientais frequentemente trabalham com fósforo total. Assim, sua utilização direta poderia gerar inconsistências metodológicas na interpretação normativa.

Já a variável `Temperature (cel)` será mantida apenas como variável contextual. Embora a temperatura influencie diversos processos físico-químicos e ecológicos da água, ela não será utilizada como critério principal na classificação inicial do rótulo.



# Definição dos Limites da CONAMA

Nesta etapa, serão definidos os limites de referência utilizados para avaliar os parâmetros selecionados com base na Resolução CONAMA nº 357/2005.

A referência adotada foi a Classe 2 para águas doces, pois ela representa uma categoria adequada ao contexto do AquaSense, envolvendo usos como abastecimento após tratamento convencional, proteção das comunidades aquáticas, recreação, irrigação, aquicultura e pesca.

Limites utilizados:

- pH: entre 6,0 e 9,0;
- Oxigênio Dissolvido: não inferior a 5 mg/L;
- DBO: até 5 mg/L;
- Nitrato: até 10 mg/L;
- Amônia: limite variável conforme o pH.

Esses limites serão utilizados para transformar os valores físico-químicos do dataset em indicadores de conformidade ambiental.

In [ ]:

# LIMITES CONAMA - ÁGUAS DOCES


conama_reference = {

    "pH (ph units)": {
        "classe_1": {"min": 6.0, "max": 9.0},
        "classe_2": {"min": 6.0, "max": 9.0},
        "classe_3": {"min": 6.0, "max": 9.0}
    },

    "Dissolved Oxygen (mg/l)": {
        "classe_1": {"min": 6.0},
        "classe_2": {"min": 5.0},
        "classe_3": {"min": 4.0}
    },

    "Biochemical Oxygen Demand (mg/l)": {
        "classe_1": {"max": 3.0},
        "classe_2": {"max": 5.0},
        "classe_3": {"max": 10.0}
    },

    "Nitrate (mg/l)": {
        "classe_1": {"max": 10.0},
        "classe_2": {"max": 10.0},
        "classe_3": {"max": 10.0}
    }
}

In [ ]:
# ESTATÍSTICAS DOS PARÂMETROS

selected_parameters = [
    "pH (ph units)",
    "Dissolved Oxygen (mg/l)",
    "Biochemical Oxygen Demand (mg/l)",
    "Nitrate (mg/l)",
    "Ammonia (mg/l)"
]

parameter_stats = df[selected_parameters].describe()

parameter_stats

,pH (ph units),Dissolved Oxygen (mg/l),Biochemical Oxygen Demand (mg/l),Nitrate (mg/l),Ammonia (mg/l)
count,2.827977e+06,2.827977e+06,2.827977e+06,2.827977e+06,2.827977e+06
mean,7.735931e+00,1.000798e+01,4.886554e+00,4.766794e+00,1.171360e+00
std,4.946871e-01,1.851052e+00,1.641363e+01,6.073781e+00,5.668927e+00
min,-1.000000e+00,0.000000e+00,-2.000000e+00,0.000000e+00,-5.000000e-03
25%,7.550000e+00,9.860000e+00,1.600000e+00,1.173000e+00,3.000000e-02
50%,7.780000e+00,1.020000e+01,2.700000e+00,4.500000e+00,5.500000e-02
75%,8.000000e+00,1.100000e+01,2.830000e+00,4.940000e+00,3.170000e-01
max,3.000000e+01,2.000000e+01,2.550000e+02,1.550000e+02,2.000000e+02


In [ ]:

# CRIAÇÃO DAS COLUNAS AUXILIARES - CLASSE 2 CONAMA
# pH: adequado se estiver entre 6 e 9
df["ph_ok"] = np.where(
    df["pH (ph units)"].between(6, 9),
    1,
    0
)

# Oxigênio dissolvido: adequado se for maior ou igual a 5 mg/L
df["od_ok"] = np.where(
    df["Dissolved Oxygen (mg/l)"] >= 5,
    1,
    0
)

# DBO: adequado se for menor ou igual a 5 mg/L
df["dbo_ok"] = np.where(
    df["Biochemical Oxygen Demand (mg/l)"] <= 5,
    1,
    0
)

# Nitrato: adequado se for menor ou igual a 10 mg/L
df["nitrate_ok"] = np.where(
    df["Nitrate (mg/l)"] <= 10,
    1,
    0
)

# Amônia: limite depende do pH
def ammonia_limit(ph):
    if ph <= 7.5:
        return 3.7
    elif ph <= 8.0:
        return 2.0
    elif ph <= 8.5:
        return 1.0
    else:
        return 0.5

df["ammonia_limit"] = df["pH (ph units)"].apply(ammonia_limit)

df["ammonia_ok"] = np.where(
    df["Ammonia (mg/l)"] <= df["ammonia_limit"],
    1,
    0
)




auxiliary_columns = [
    "ph_ok",
    "od_ok",
    "dbo_ok",
    "nitrate_ok",
    "ammonia_ok"
]

auxiliary_summary = []

for col in auxiliary_columns:
    counts = df[col].value_counts()

    auxiliary_summary.append({
        "Coluna auxiliar": col,
        "Dentro do limite (1)": counts.get(1, 0),
        "Fora do limite (0)": counts.get(0, 0),
        "Dentro (%)": round((counts.get(1, 0) / len(df)) * 100, 2),
        "Fora (%)": round((counts.get(0, 0) / len(df)) * 100, 2)
    })

auxiliary_summary = pd.DataFrame(auxiliary_summary)

auxiliary_summary

,Coluna auxiliar,Dentro do limite (1),Fora do limite (0),Dentro (%),Fora (%)
0,ph_ok,2802289,25688,99.09,0.91
1,od_ok,2722312,105665,96.26,3.74
2,dbo_ok,2444632,383345,86.44,13.56
3,nitrate_ok,2557939,270038,90.45,9.55
4,ammonia_ok,2609136,218841,92.26,7.74


# Construção do Score Ambiental

Após a criação das colunas auxiliares, foi desenvolvido um score ambiental baseado na quantidade de critérios da Classe 2 da CONAMA atendidos por cada amostra.

O objetivo desse score é representar o nível de conformidade ambiental da água de forma gradual, permitindo a construção de um rótulo multiclasses mais robusto e interpretável.

Diferentemente de uma classificação binária simples ("adequada" ou "inadequada"), o score ambiental permite identificar diferentes níveis de qualidade da água, preservando maior riqueza de informação para os modelos de Machine Learning.

Cada parâmetro dentro do limite estabelecido pela CONAMA recebe valor `1`, enquanto parâmetros fora do limite recebem valor `0`.

O score final corresponde à soma das colunas auxiliares:
- pH;
- Oxigênio Dissolvido;
- DBO;
- Nitrato;
- Amônia.

Assim, quanto maior o score, maior o nível de conformidade ambiental da amostra.

In [ ]:

df["environmental_score"] = (

    df["ph_ok"] +

    df["od_ok"] +

    df["dbo_ok"] +

    df["nitrate_ok"] +

    df["ammonia_ok"]

)



df["environmental_score"].value_counts().sort_index()

,count
environmental_score,
0,1
1,533
2,27175
3,191243
4,537429
5,2071596


In [ ]:

# CRIAÇÃO DO RÓTULO CONAMA_STATUS

def classify_conama_status(score):
    if score == 5:
        return "Adequada"
    elif score >= 3:
        return "Atenção"
    else:
        return "Crítica"

df["conama_status"] = df["environmental_score"].apply(classify_conama_status)

# Visualizar distribuição do rótulo
df["conama_status"].value_counts()

,count
conama_status,
Adequada,2071596
Atenção,728672
Crítica,27709


In [ ]:
# Distribuição percentual
df["conama_status"].value_counts(normalize=True) * 100

,proportion
conama_status,
Adequada,73.253637
Atenção,25.766546
Crítica,0.979817


In [ ]:
pd.crosstab(
    df["environmental_score"],
    df["conama_status"]
)

conama_status,Adequada,Atenção,Crítica
environmental_score,,,
0,0,0,1
1,0,0,533
2,0,0,27175
3,0,191243,0
4,0,537429,0
5,2071596,0,0


O rótulo multiclasses foi construído com base na quantidade de critérios da Classe 2 da CONAMA atendidos por cada amostra, permitindo representar diferentes níveis de conformidade ambiental.

In [ ]:
output_path = "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/water_quality_rotulado_3.parquet"

df.to_parquet(
    output_path,
    index=False
)

print("Dataset rotulado salvo com sucesso!")

Dataset rotulado salvo com sucesso!


In [ ]:
rotulado_path = "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/water_quality_rotulado_3.parquet"

df_rotulado = pd.read_parquet(rotulado_path)

print("Dataset rotulado carregado com sucesso!")

Dataset rotulado carregado com sucesso!


In [ ]:
df_rotulado.columns.tolist()

['Country',
 'Area',
 'Waterbody Type',
 'Date',
 'Ammonia (mg/l)',
 'Biochemical Oxygen Demand (mg/l)',
 'Dissolved Oxygen (mg/l)',
 'Orthophosphate (mg/l)',
 'pH (ph units)',
 'Temperature (cel)',
 'Nitrogen (mg/l)',
 'Nitrate (mg/l)',
 'CCME_Values',
 'CCME_WQI',
 'ph_ok',
 'od_ok',
 'dbo_ok',
 'nitrate_ok',
 'ammonia_limit',
 'ammonia_ok',
 'environmental_score',
 'conama_status']

In [ ]:
df_rotulado.iloc[:, -10:].head()

,CCME_Values,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status
0,100.0,Excellent,1,1,1,0,1.0,1,4,Atenção
1,100.0,Excellent,1,1,1,0,2.0,1,4,Atenção
2,100.0,Excellent,1,1,1,0,3.7,1,4,Atenção
3,100.0,Excellent,1,1,1,0,2.0,1,4,Atenção
4,100.0,Excellent,1,0,1,0,3.7,1,3,Atenção


In [ ]:
df_rotulado.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2827977 entries, 0 to 2827976
Data columns (total 22 columns):
 #   Column                            Dtype  
---  ------                            -----  
 0   Country                           object 
 1   Area                              object 
 2   Waterbody Type                    object 
 3   Date                              object 
 4   Ammonia (mg/l)                    float64
 5   Biochemical Oxygen Demand (mg/l)  float64
 6   Dissolved Oxygen (mg/l)           float64
 7   Orthophosphate (mg/l)             float64
 8   pH (ph units)                     float64
 9   Temperature (cel)                 float64
 10  Nitrogen (mg/l)                   float64
 11  Nitrate (mg/l)                    float64
 12  CCME_Values                       float64
 13  CCME_WQI                          object 
 14  ph_ok                             int64  
 15  od_ok                             int64  
 16  dbo_ok                            in

In [ ]:
df_rotulado.head()

,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,CCME_Values,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status
0,Canada,SE649035-145565,River,12-01-1974,0.059248,1.30,8.1500,0.011917,8.07500,9.885,...,100.0,Excellent,1,1,1,0,1.0,1,4,Atenção
1,Canada,SE649035-145565,River,12-01-1975,0.039821,1.38,7.8000,0.009417,7.73333,10.150,...,100.0,Excellent,1,1,1,0,2.0,1,4,Atenção
2,Canada,SE649035-145565,River,12-01-1976,0.031341,2.23,7.8000,0.011000,7.46667,10.235,...,100.0,Excellent,1,1,1,0,3.7,1,4,Atenção
3,Canada,SE649035-145565,River,12-01-1977,0.020501,1.61,8.1500,0.012333,7.78333,11.116,...,100.0,Excellent,1,1,1,0,2.0,1,4,Atenção
4,Canada,SE649035-145565,River,12-01-1978,0.020023,1.64,4.3708,0.006182,7.10000,7.068,...,100.0,Excellent,1,0,1,0,3.7,1,3,Atenção


# Construção da Amostra Representativa

Para gerar a amostra utilizada nos experimentos, foi aplicada uma estratégia de amostragem proporcional por país (`Country`).

O objetivo dessa abordagem foi preservar a distribuição geográfica presente no dataset original, evitando que determinados países ficassem super-representados ou sub-representados na nova base.

A amostra foi criada utilizando uma fração de 5% dos dados de cada país.

Além disso, foi utilizado:

```python
random_state=42
```

Isso garante que a amostra gerada seja sempre reproduzível, ou seja, sempre que o código for executado, os mesmos registros serão selecionados.

Após a geração da amostra, foram realizadas validações estatísticas comparando:

* distribuição dos países;
* distribuição das classes;
* estatísticas descritivas;
* comportamento geral das variáveis;

entre:

* dataset original;
* dataset amostrado.

Os resultados mostraram que a amostra conseguiu preservar adequadamente o comportamento estatístico do dataset original, tornando-se válida para utilização nos experimentos de Machine Learning.

A partir desta etapa, passou-se a utilizar:

```python
df_amostra
```

como base principal para:


* treinamento dos modelos;
* avaliação das métricas;
* experimentos comparativos.


In [ ]:
df_rotulado = pd.read_parquet(
    "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/water_quality_rotulado_3.parquet"
)


In [ ]:
df_amostra = (
    df_rotulado.groupby("Country", group_keys=False)
      .sample(frac=0.05, random_state=42)
      .reset_index(drop=True)
)

In [ ]:
print(df_amostra.shape)

(141399, 22)


#### Validação Estatística da Amostra

Após a criação da amostra proporcional estratificada por país, foi realizada a comparação entre a distribuição das classes do dataset original e da amostra gerada.

Os resultados demonstraram que a amostra preservou de forma consistente a distribuição do rótulo `conama_status`, mantendo proporções extremamente próximas às observadas no conjunto completo de dados.

Isso indica que a amostra permaneceu representativa do comportamento geral do dataset original, permitindo sua utilização nos experimentos de Machine Learning sem perda significativa de informação estatística.

In [ ]:
df_rotulado["Country"].value_counts(normalize=True)
df_amostra["Country"].value_counts(normalize=True)


,proportion
Country,
England,0.752905
USA,0.146331
Ireland,0.083105
China,0.016266
Canada,0.001393


In [ ]:
df_rotulado["conama_status"].value_counts(normalize=True)
df_amostra["conama_status"].value_counts(normalize=True)

,proportion
conama_status,
Adequada,0.731752
Atenção,0.258467
Crítica,0.009781


In [ ]:
print("Original:")
print(df_rotulado["Country"].value_counts(normalize=True))

print("\nAmostra:")
print(df_amostra["Country"].value_counts(normalize=True))

Original:
Country
England    0.752905
USA        0.146329
Ireland    0.083105
China      0.016265
Canada     0.001396
Name: proportion, dtype: float64

Amostra:
Country
England    0.752905
USA        0.146331
Ireland    0.083105
China      0.016266
Canada     0.001393
Name: proportion, dtype: float64


In [ ]:
print("Original:")
print(df_rotulado["conama_status"].value_counts(normalize=True))

print("\nAmostra:")
print(df_amostra["conama_status"].value_counts(normalize=True))

Original:
conama_status
Adequada    0.732536
Atenção     0.257665
Crítica     0.009798
Name: proportion, dtype: float64

Amostra:
conama_status
Adequada    0.731752
Atenção     0.258467
Crítica     0.009781
Name: proportion, dtype: float64


In [ ]:
print("Original:")
print(df_rotulado["CCME_Values"].describe())

print("\nAmostra:")
print(df_amostra["CCME_Values"].describe())

Original:
count    2.827977e+06
mean     8.504668e+01
std      1.764665e+01
min      3.130414e+01
25%      7.715349e+01
50%      9.059609e+01
75%      1.000000e+02
max      1.000000e+02
Name: CCME_Values, dtype: float64

Amostra:
count    141399.000000
mean         85.038443
std          17.650268
min          34.607162
25%          77.230685
50%          90.543682
75%         100.000000
max         100.000000
Name: CCME_Values, dtype: float64


In [ ]:
df_amostra.to_parquet(
    "/content/drive/MyDrive/EDA_AquaSense/amostra_rotulada_3.parquet",
    index=False
)

> A amostra manteve distribuição estatística
muito próxima ao dataset original,
preservando média, dispersão
e quartis dos indicadores analisados,
o que indica representatividade adequada
para os experimentos de Machine Learning.

# Análise Experimental dos Modelos de Machine Learning

Depois da construção do `conama_status`, a próxima etapa do projeto passou a ser a análise experimental dos modelos de Machine Learning.

Essa etapa é extremamente importante porque, na prática, não basta apenas treinar um algoritmo e observar a acurácia final. O objetivo da análise experimental é entender como o modelo se comporta em diferentes cenários, quais variáveis realmente ajudam na previsão, como o balanceamento impacta os resultados e até que ponto o modelo consegue generalizar padrões ambientais sem apenas decorar os dados de treino.

Como o dataset possui um forte desbalanceamento entre as classes, principalmente pela grande quantidade de amostras classificadas como `Excelente`, existe o risco do modelo aprender padrões enviesados. Nesse cenário, o algoritmo poderia simplesmente prever a classe majoritária na maior parte das vezes e ainda assim obter uma acurácia aparentemente alta, mesmo apresentando baixo desempenho para identificar amostras críticas.

Por isso, a análise experimental foi estruturada em diferentes cenários de teste, permitindo comparar:

- modelos com e sem balanceamento;
- impacto da adição de novas variáveis;
- influência de variáveis categóricas;
- comportamento das métricas de treino e teste;
- presença de overfitting;
- capacidade de generalização do modelo.

Além disso, os experimentos também ajudam a entender se determinadas variáveis conseguem fornecer informações ambientais relevantes mesmo sem terem sido utilizadas diretamente na construção do rótulo `conama_status`.

Outro ponto importante é que o objetivo do projeto não é apenas obter uma alta acurácia, mas construir um modelo coerente ambientalmente, interpretável e metodologicamente defensável.

Por esse motivo, todos os experimentos seguem uma mesma estrutura padronizada:

1. Definição das variáveis de entrada (`X`) e da variável alvo (`y`);
2. Separação entre treino e teste;
3. Aplicação ou não de técnicas de balanceamento;
4. Treinamento do modelo;
5. Avaliação utilizando métricas;
6. Comparação entre os resultados obtidos.

Essa abordagem permite analisar como pequenas alterações na estrutura do treinamento influenciam diretamente o comportamento do modelo e a qualidade das previsões realizadas.



---


> Ignorar essa parte, pois foi criada apenas para testes iniciais.

> Os experimentos **corretos** estão divididos em notebooks separados.